In [0]:
import requests
import pandas as pd
from pyspark.sql import functions as F


def get_history(start, end, country="be"):      # get all back history from 01/2021 to 10/2026
    r = requests.get(
        "https://api.energy-charts.info/v2/public_power",
        params={"country": country,"start": start, "end": end},
        timeout=30,
    )
    print("Statut requete:", r.status_code)
    data = r.json()
    data = data["data"]
    df = pd.json_normalize(data)
    df.columns = [c.replace("values.", "") for c in df.columns]     # we remove the "values." created by json_normalize(based on json architecture)
    display(df.head(25))
    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)     
    return df

df = get_history("2021-01-01", "2026-10-16")
print(df.shape)
df_all = df.sort_values("timestamp")

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.energy")

# convert df to sparkdf
sdf = spark.createDataFrame(df)
print("line number to save :", df.shape)
# we create the column ingested_at which will get us the log of modification 
sdf = sdf.withColumn("ingested_at", F.current_timestamp())        
sdf = sdf.withColumn("source", F.lit("energy-charts/v2/public_power"))

# save as delta, overwrite = recreate delta if already exists, name = bronze_public_power
sdf.write.format("delta").mode("overwrite").saveAsTable("workspace.energy.bronze_public_power")

In [0]:
%sql
SELECT count(*) as nb_lignes,
min(timestamp) as debut,
max(timestamp) as fin
from workspace.energy.bronze_public_power;


SELECT * 
FROM workspace.energy.bronze_public_power
ORDER BY `timestamp`
limit 10

